# imports

In [ ]:
from alphagenome import colab_utils
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.interpretation import ism
from alphagenome.models import dna_client
from alphagenome.models import variant_scorers
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns


# import ism variant scores

In [ ]:
df = pd.read_csv("/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_ism_pre_region1.csv")
chromosome = 'chr1'

In [ ]:
variants = [
    genome.Variant(
        chromosome=chromosome,
        position=int(row.position),
        reference_bases=row.ref,
        alternate_bases=row.alt,
    )
    for row in df.itertuples()
]
scores = df["delta_score"].tolist()

# Convert 1-based CSV coords -> 0-based interval
ism_interval = genome.Interval(
    chromosome,
    start=int(df["position"].min()) - 1,
    end=int(df["position"].max()),
)

In [ ]:
ism_result = ism.ism_matrix(scores, variants, interval=ism_interval)

plot_components.plot(
    [plot_components.SeqLogo(
        scores=ism_result,
        scores_interval=ism_interval,
        ylabel="LMNA ISM (region 1)",
    )],
    interval=ism_interval,
    fig_width=25,  # wider for 1000 bp
)
plt.show()

In [ ]:
summary = df.groupby("position")["delta_score"].max()
plt.figure(figsize=(14, 3))
plt.plot(summary.index, summary.values, linewidth=0.8)
plt.xlabel("Genomic position (hg38)")
plt.ylabel("Max |delta_score|")
plt.title("ISM importance by position")
plt.tight_layout()
plt.show()

In [ ]:

pivot = df.pivot(index="position", columns="alt", values="delta_score")
# subsample if too dense
step = max(1, len(pivot) // 200)
plt.figure(figsize=(6, 10))
sns.heatmap(pivot.iloc[::step], cmap="RdBu_r", center=0)
plt.ylabel("Position (subsampled)")
plt.xlabel("Alternate base")
plt.show()

In [ ]:
CCRE_BED = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_cCREs_chr1_156064711_156190081.bed"

ccre = pd.read_csv(
    CCRE_BED,
    sep="\t",
    comment="t",   # skip the "track name=..." header line
    header=None,
    names=[
        "chrom", "start", "end", "name", "score",
        "strand", "thickStart", "thickEnd", "itemRgb",
    ],
)

# Optional: map ENCODE cCRE colors to labels
CCRE_COLOR_MAP = {
    "255,205,0": "dELS", #Distal Enhancer-Like Sequence
    "255,167,0": "pELS", #Proximal Enhancer-Like Sequence
    "255,0,0": "CA", #CpG Island
    "255,170,170": "CA-CTCF", #CpG Island with CTCF binding
    "216,118,236": "CA-TF", #CpG Island with TF binding
    "6,218,147": "CA-CTCF", #CpG Island with CTCF binding
    "0,176,240": "DNase-H3K4me3", #DNase hypersensitive site with H3K4me3 binding
}
ccre["ccre_class"] = ccre["itemRgb"].map(CCRE_COLOR_MAP).fillna("other")

ccre.head()

In [ ]:
summary = df.groupby("position")["delta_score"].max()

plot_start = int(summary.index.min())
plot_end = int(summary.index.max())

# Keep cCREs overlapping the ISM window
ccre_in_view = ccre[
    (ccre["chrom"] == chromosome) &
    (ccre["end"] >= plot_start) &
    (ccre["start"] <= plot_end)
].copy()

print(f"Plotting {len(ccre_in_view)} cCREs in ISM window")
ccre_in_view[["start", "end", "name", "ccre_class"]]

In [ ]:
def rgb_str_to_tuple(s):
    return tuple(int(x) / 255 for x in s.split(","))

fig, ax = plt.subplots(figsize=(14, 3))

# 1) cCRE spans first (so they sit behind the line)
for _, row in ccre_in_view.iterrows():
    ax.axvspan(
        row["start"], row["end"],
        color=rgb_str_to_tuple(row["itemRgb"]),
        alpha=0.25,
        linewidth=0,
        zorder=1,
    )

# 2) ISM line on top
ax.plot(summary.index, summary.values, linewidth=0.8, color="black", zorder=2)

ax.set_xlabel("Genomic position (hg38)")
ax.set_ylabel("Max delta_score")
ax.set_title("ISM importance by position with cCREs")
ax.set_xlim(plot_start, plot_end)
plt.tight_layout()
plt.show()

In [ ]:
def plot_ism_with_ccre_track(summary, ccre_df, chromosome, plot_start, plot_end):
    ccre_in_view = ccre_df[
        (ccre_df["chrom"] == chromosome) &
        (ccre_df["end"] >= plot_start) &
        (ccre_df["start"] <= plot_end)
    ]

    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 4), sharex=True,
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.05},
    )

    # Top: ISM
    ax1.plot(summary.index, summary.values, linewidth=0.8, color="black")
    ax1.set_ylabel("Max delta_score")
    ax1.set_title("ISM importance with cCRE annotations")
    ax1.set_xlim(plot_start, plot_end)

    # Bottom: cCRE rectangles
    for i, (_, row) in enumerate(ccre_in_view.iterrows()):
        ax2.broken_barh(
            [(row["start"], row["end"] - row["start"])],
            (i * 0.8, 0.6),
            facecolors=[rgb_str_to_tuple(row["itemRgb"])],
            edgecolor="none",
        )

    ax2.set_yticks([])
    ax2.set_xlabel("Genomic position (hg38)")
    ax2.set_ylabel("cCREs")
    ax2.set_xlim(plot_start, plot_end)

    plt.tight_layout()
    plt.show()

plot_ism_with_ccre_track(summary, ccre, chromosome, plot_start, plot_end)

In [ ]:
FANTOM5_BED = "/Users/coraalbers/Documents/BSGP/Lancaster_rotation/data/lmna_FANTOM5_chr1_156064711_156190081.bed"

fantom = pd.read_csv(
    FANTOM5_BED,
    sep="\t",
    comment="t",
    header=None,
    names=[
        "chrom", "start", "end", "name", "score", "strand",
        "thickStart", "thickEnd", "itemRgb",
        "blockCount", "blockSizes", "blockStarts",
    ],
)

def exons_in_window(row, win_start, win_end):
    """Return exon intervals from a BED12 row, clipped to [win_start, win_end]."""
    chrom_start = int(row["start"])
    sizes = [int(x) for x in str(row["blockSizes"]).rstrip(",").split(",") if x]
    rel_starts = [int(x) for x in str(row["blockStarts"]).rstrip(",").split(",") if x]

    exons = []
    for size, rel in zip(sizes, rel_starts):
        s = chrom_start + rel
        e = s + size
        if e >= win_start and s <= win_end:
            exons.append((max(s, win_start), min(e, win_end)))
    return exons

fantom_in_view = fantom[
    (fantom["chrom"] == chromosome) &
    (fantom["end"] >= plot_start) &
    (fantom["start"] <= plot_end)
].copy()

fantom_exons = []
for _, row in fantom_in_view.iterrows():
    for start, end in exons_in_window(row, plot_start, plot_end):
        fantom_exons.append({
            "start": start,
            "end": end,
            "name": row["name"],
            "strand": row["strand"],
        })

fantom_exons = pd.DataFrame(fantom_exons).drop_duplicates(subset=["start", "end"])
print(f"Plotting {len(fantom_exons)} FANTOM5 exon intervals in ISM window")
fantom_exons

In [ ]:
FANTOM5_COLOR = (0.85, 0.33, 0.10)


def plot_ism_with_annotation_tracks(
    summary,
    ccre_df,
    fantom_exons_df,
    chromosome,
    plot_start,
    plot_end,
):
    ccre_in_view = ccre_df[
        (ccre_df["chrom"] == chromosome) &
        (ccre_df["end"] >= plot_start) &
        (ccre_df["start"] <= plot_end)
    ]

    fig, (ax1, ax2, ax3) = plt.subplots(
        3, 1, figsize=(14, 5.5), sharex=True,
        gridspec_kw={"height_ratios": [3, 1, 1], "hspace": 0.08},
    )

    # Top: ISM line plot
    ax1.plot(summary.index, summary.values, linewidth=0.8, color="black")
    ax1.set_ylabel("Max delta_score")
    ax1.set_title("ISM importance with cCRE and FANTOM5 annotations")
    ax1.set_xlim(plot_start, plot_end)

    # Middle: cCRE track
    for i, (_, row) in enumerate(ccre_in_view.iterrows()):
        ax2.broken_barh(
            [(row["start"], row["end"] - row["start"])],
            (i * 0.8, 0.6),
            facecolors=[rgb_str_to_tuple(row["itemRgb"])],
            edgecolor="none",
        )
    ax2.set_yticks([])
    ax2.set_ylabel("cCREs")
    ax2.set_xlim(plot_start, plot_end)

    # Bottom: FANTOM5 exon track
    for i, (_, row) in enumerate(fantom_exons_df.iterrows()):
        ax3.broken_barh(
            [(row["start"], row["end"] - row["start"])],
            (i * 0.8, 0.6),
            facecolors=[FANTOM5_COLOR],
            edgecolor="none",
        )
    ax3.set_yticks([])
    ax3.set_xlabel("Genomic position (hg38)")
    ax3.set_ylabel("FANTOM5")
    ax3.set_xlim(plot_start, plot_end)

    plt.tight_layout()
    plt.show()


plot_ism_with_annotation_tracks(
    summary, ccre, fantom_exons, chromosome, plot_start, plot_end
)